In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

# print("Path to dataset files:", path)

In [1]:
import os
import librosa
import librosa.display
from librosa.feature import melspectrogram
import matplotlib.pyplot as plt
import numpy as np

current_dir: str = os.getcwd()
ravdess_path = os.path.join(os.getcwd(), "..", "data", "ravdess")

d:\Coding\Projects\private_PRJ\speech-emotion-recognition-ravdess\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
class RAVDESSDataset:
    def __init__(self, base_path):
        self.base_path = base_path

    def load_audio(self, num_actors: int = 24):
        """
        load each audio (end with .wav) in each actor folder (start with Actor)
        save audio and audio data

        Args:
            num_actors (int, optional): The number of actors to load audio from. Defaults to 24.

        Returns:
            audio_files (list): A list of file paths for the loaded audio files.
            audio_data (list): A list of tuples containing the audio data and sample rate for each loaded audio file.
        """
        audio_files = []
        audio_data = []
        actor_folders = []

        all_actors = sorted(
            [d for d in os.listdir(self.base_path) if d.startswith("Actor")]
        )

        all_actors = all_actors[:num_actors]

        # if all_actors.__len__() != 0:
        #     return "No actor folders or audio found in the specified base path."

        for actor in all_actors:
            actor_path = os.path.join(self.base_path, actor)

            for audio in os.listdir(actor_path):
                if audio.endswith(".wav"):
                    audio_files.append(os.path.join(actor_path, audio))
                    y, sr = librosa.load(os.path.join(actor_path, audio), sr=None)
                    audio_data.append((y, sr))
            actor_folders.append(actor)

        return audio_files, audio_data, actor_folders

    def convert_to_spectrogram(self, audio_data: list):
        """
        Convert the loaded audio data to spectrograms.

        Args:
            audio_data (list): A list of tuples containing the audio data and sample rate for each audio file.

        Returns:
            list: A list of spectrograms.
        """
        spectrograms = []
        for y, sr in audio_data:
            S = melspectrogram(y=y, sr=sr, n_mels=128)
            S_dB = librosa.power_to_db(S, ref=np.max)

            S_dB = (S_dB - np.mean(S_dB)) / (np.std(S_dB) + 1e-6)

            spectrograms.append(S_dB)
        return spectrograms

    def create_spectrogram_result(
        self, spectrograms: list, audio_files: list, result_folder: str, sr=22050
    ):
        if not os.path.exists(result_folder):
            os.makedirs(result_folder)

        for spectrogram, audio_file in zip(spectrograms, audio_files):
            actor_name = os.path.basename(os.path.dirname(audio_file))
            if not os.path.exists(os.path.join(result_folder, actor_name)):
                os.makedirs(os.path.join(result_folder, actor_name))
            result_path = os.path.join(result_folder, actor_name)
            spectrogram_filename = (
                os.path.splitext(os.path.basename(audio_file))[0] + "_spectrogram.png"
            )
            spectrogram_filepath = os.path.join(result_path, spectrogram_filename)

            self.save_mel_spectrogram(spectrogram, spectrogram_filepath, sr=sr)

            print(
                f"Saved spectrogram for {os.path.basename(spectrogram_filename)} at {actor_name}"
            )

    def save_mel_spectrogram(self, spectrogram, spectrogram_filepath, sr):
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(spectrogram, sr=sr, x_axis="time", y_axis="mel")
        plt.colorbar(format="%+2.0f dB")
        plt.title("Mel Spectrogram")
        plt.tight_layout()
        plt.savefig(spectrogram_filepath)
        plt.close()

    def __len__(self):
        return len(os.listdir(self.base_path))

**Load audio files, audio data**

In [13]:
dataset = RAVDESSDataset(ravdess_path)

audio_files, audio_data, actor_folders = dataset.load_audio()

print(f"Sample rate: {audio_data[0][1]}")

Sample rate: 48000


**Convert dataset to mel spectrogram**

In [14]:
dataset.create_spectrogram_result(dataset.convert_to_spectrogram(audio_data), audio_files, os.path.join(os.getcwd(), "result"), 48000)

Saved spectrogram for 03-01-01-01-01-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-01-01-01-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-01-01-02-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-01-01-02-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-01-01-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-01-01-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-01-02-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-01-02-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-02-01-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-02-01-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-02-02-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-02-02-02-02-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-03-01-01-01-01_spectrogram.png at Actor_01
Saved spectrogram for 03-01-03-01-01-02-01_spectrogram.png at Actor_01
Saved 